#### 0. Realizar a limpeza dos diretórios para evitar falhas de execução

In [1]:
%%bash
rm -rf ../../data/2-spec/*

#### 1. Realizar a importação dos módulos

In [2]:
from dotenv import load_dotenv
import duckdb
import os

#### 2. Realizar a conexão com o banco de dados Postgress

In [3]:
load_dotenv()

con = duckdb.connect()

con.execute("INSTALL postgres;")

con.execute("LOAD postgres;")

con_str = (
    f"host={os.getenv('PG_HOST')} "
    f"dbname={os.getenv('PG_DB')} "
    f"user={os.getenv('PG_USER')} "
    f"password={os.getenv('PG_PASSWORD')} "
    f"port={os.getenv('PG_PORT')}"
)

con.execute(f"""ATTACH '{con_str}' AS pg (TYPE postgres);""")

#### 3. Realizar a criação da tabela flat (ou não modelada) no banco de dados

In [4]:
con.execute(f"""
    DROP TABLE IF EXISTS pg.public.tb_spec_viagem;
            
    CREATE TABLE pg.public.tb_spec_viagem 
    (
        identificador_empresa TEXT,
        identificador_base_despacho TEXT,
        identificador_base_origem TEXT,
        anomes_solicitacao TEXT,
        data_solicitacao TIMESTAMP,
        nome_dia_semana_solicitacao TEXT,
        hora_solicitacao TEXT,
        nome_periodo_solicitacao TEXT,
        anomes_chegada_motorista TEXT,
        data_chegada_motorista TIMESTAMP,
        nome_dia_semana_chegada TEXT,
        hora_chegada_motorista TEXT,
        nome_periodo_chegada TEXT,
        anomes_inicio_viagem TEXT,
        data_inicio_viagem TIMESTAMP,
        nome_dia_semana_inicio_viagem TEXT,
        hora_inicio_viagem TEXT,
        nome_periodo_inicio_viagem TEXT,
        diferenca_chegada_motorista_fim_viagem BIGINT,
        anomes_fim_viagem TEXT,
        data_fim_viagem TIMESTAMP,
        nome_dia_semana_fim_viagem TEXT,
        hora_fim_viagem TEXT,
        nome_periodo_fim_viagem TEXT,
        diferenca_inicio_viagem_fim_viagem BIGINT,
        identificador_localizacao_embarque DOUBLE PRECISION,
        identificador_localizacao_desembarque DOUBLE PRECISION,
        quantidade_milhas_viagem DOUBLE PRECISION,
        quantidade_segundos_viagem DOUBLE PRECISION,
        valor_velocidade_media_viagem DOUBLE PRECISION,
        valor_tarifa_base_passageiro DOUBLE PRECISION,
        valor_pedagios DOUBLE PRECISION,
        valor_fundo_black_car DOUBLE PRECISION,
        valor_imposto_vendas DOUBLE PRECISION,
        valor_taxa_congestionamento DOUBLE PRECISION,
        valor_taxa_aeroporto DOUBLE PRECISION,
        valor_gorjeta DOUBLE PRECISION,
        valor_pagamento_motorista DOUBLE PRECISION,
        valor_receita_total_viagem DOUBLE PRECISION,
        indicador_corrida_compartilhada_solicitada TEXT,
        indicador_corrida_compartilhada_realizada TEXT,
        indicador_corrida_mta TEXT,
        indicador_veiculo_acessivel_solicitado TEXT,
        indicador_veiculo_acessivel_realizado TEXT,
        valor_taxa_congestionamento_cb DOUBLE PRECISION
    );
"""
)

#### 2.1. Realizar a inserção dos dados da tabela flat no banco de dados

In [5]:
input_path = "../../data/1-sot/tb-sot-fhvhv-trip-data-unified.parquet"

con.execute(f"""
    INSERT INTO 
        pg.public.tb_spec_viagem
    SELECT 
        *
    FROM 
        read_parquet('{input_path}');
"""
)

#### 3. Realizar a modelagem da dimensão data (completa, anomes, ano, mes, dia, e dia da semana)

In [6]:
con.execute(f"""
    DROP TABLE IF EXISTS pg.public.tb_spec_dim_data;
            
    CREATE TABLE pg.public.tb_spec_dim_data
    (
        data TIMESTAMP,
        anomes TEXT,
        ano TEXT,
        mes TEXT,
        dia TEXT,
        dia_semana TEXT
    );
"""
)

#### 3.1. Realizar a inserção dos dados da dimensão data

In [7]:
con.execute(f"""
    INSERT INTO 
        pg.public.tb_spec_dim_data
    SELECT DISTINCT
        data AS data,
        SUBSTRING(CAST(data AS TEXT), 1, 7) AS anomes,
        SUBSTRING(CAST(data AS TEXT), 1, 4) AS ano,
        SUBSTRING(CAST(data AS TEXT), 6, 2) AS mes,
        SUBSTRING(CAST(data AS TEXT), 9, 2) AS dia,
        dia_semana as dia_semana
    FROM
        (
            SELECT DISTINCT data_solicitacao AS data, nome_dia_semana_solicitacao AS dia_semana FROM pg.public.tb_spec_viagem
            UNION ALL
            SELECT DISTINCT data_chegada_motorista AS data, nome_dia_semana_chegada AS dia_semana FROM pg.public.tb_spec_viagem
            UNION ALL
            SELECT DISTINCT data_inicio_viagem AS data, nome_dia_semana_inicio_viagem AS dia_semana FROM pg.public.tb_spec_viagem
            UNION ALL
            SELECT DISTINCT data_fim_viagem AS data, nome_dia_semana_fim_viagem AS dia_semana FROM pg.public.tb_spec_viagem
        );
"""
)

#### 4. Realizar a modelagem da dimensão empresa

- [Clique aqui](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_hvfhs.pdf) para conhecer o domínio de empresas.
- [Clique aqui](https://data.cityofnewyork.us/api/views/eccv-9dzr/rows.pdf?app_token=U29jcmF0YS0td2VraWNrYXNz0) para conhecer o domínio de bases.

In [8]:
con.execute(f"""
    DROP TABLE IF EXISTS pg.public.tb_spec_dim_empresa;
            
    CREATE TABLE pg.public.tb_spec_dim_empresa
    (
        identificador_empresa TEXT,
        nome_empresa TEXT,
        identificador_base TEXT,
        nome_base TEXT
    );
"""
)

#### 4.1. Realizar a inserção dos dados da dimensão empresa

In [9]:
con.execute(f"""
    INSERT INTO 
        pg.public.tb_spec_dim_empresa
    SELECT DISTINCT
        identificador_empresa,
        CASE 
            WHEN identificador_empresa = 'HV0002' THEN 'Juno'
            WHEN identificador_empresa = 'HV0003' THEN 'Uber'
            WHEN identificador_empresa = 'HV0004' THEN 'Via'
            WHEN identificador_empresa = 'HV0005' THEN 'Lyft'
            ELSE 'Domínio não cadastrado' 
        END AS nome_empresa,
        identificador_base,
        CASE 
            WHEN identificador_base = 'B00887' THEN 'Dial 7 Car & Limousine Service INC'
            WHEN identificador_base = 'B02026' THEN 'A Ride for All LLC'
            WHEN identificador_base = 'B03404' THEN 'Uber Usa LCC'
            WHEN identificador_base = 'B03406' THEN 'Tri City LLC'
            ELSE 'Domínio não cadastrado' 
        END AS nome_base,
    FROM
        (
            SELECT DISTINCT identificador_empresa AS identificador_empresa, identificador_base_despacho AS identificador_base FROM pg.public.tb_spec_viagem
            UNION ALL
            SELECT DISTINCT identificador_empresa AS identificador_empresa, identificador_base_origem AS identificador_base FROM pg.public.tb_spec_viagem
        );
"""
)

#### 5. Realizar a modelagem da dimensão hora

In [10]:
con.execute(f"""
    DROP TABLE IF EXISTS pg.public.tb_spec_dim_hora;
            
    CREATE TABLE pg.public.tb_spec_dim_hora
    (
        hora TEXT,
        periodo TEXT
    );
"""
)

#### 5.1. Realizar a inserção dos dados da dimensão hora

In [11]:
con.execute(f"""
    INSERT INTO 
        pg.public.tb_spec_dim_hora
    SELECT DISTINCT
        CAST(SUBSTRING(hora, 1, 2) AS TEXT) AS hora,
        CASE 
            WHEN SUBSTRING(CAST(hora AS TEXT), 1, 2) BETWEEN '00' AND '06' THEN 'a. Madrugada'
            WHEN SUBSTRING(CAST(hora AS TEXT), 1, 2) BETWEEN '07' AND '12' THEN 'b. Manhã'
            WHEN SUBSTRING(CAST(hora AS TEXT), 1, 2) BETWEEN '13' AND '18' THEN 'c. Tarde'
            WHEN SUBSTRING(CAST(hora AS TEXT), 1, 2) BETWEEN '19' AND '23' THEN 'd. Noite'
        END AS periodo,
    FROM
        (
            SELECT DISTINCT hora_solicitacao AS hora FROM pg.public.tb_spec_viagem
            UNION ALL
            SELECT DISTINCT hora_chegada_motorista AS hora FROM pg.public.tb_spec_viagem
            UNION ALL
            SELECT DISTINCT hora_inicio_viagem AS hora FROM pg.public.tb_spec_viagem
            UNION ALL
            SELECT DISTINCT hora_fim_viagem AS hora FROM pg.public.tb_spec_viagem
        );
"""
)

#### 6. Realizar a modelagem da dimensão flag

In [12]:
con.execute(f"""
    DROP TABLE IF EXISTS pg.public.tb_spec_dim_flag;
            
    CREATE TABLE pg.public.tb_spec_dim_flag
    (
        flag TEXT,
        descricao_flag TEXT
    );
"""
)

#### 6.1. Realizar a inserção dos dados da dimensão flag

In [13]:
con.execute(f"""
    INSERT INTO 
        pg.public.tb_spec_dim_flag
    VALUES
        ('Y', 'Sim'),
        ('N', 'Não');
"""
)

#### 7. Realizar a modelagem da fato viagem

In [14]:
con.execute(f"""
    DROP TABLE IF EXISTS pg.public.tb_spec_fat_viagem;
            
    CREATE TABLE pg.public.tb_spec_fat_viagem
    (
        identificador_empresa TEXT,
        identificador_base_despacho TEXT,
        identificador_base_origem TEXT,
        data_solicitacao TIMESTAMP,
        hora_solicitacao TEXT,
        data_chegada_motorista TIMESTAMP,
        hora_chegada_motorista TEXT,
        data_inicio_viagem TIMESTAMP,
        hora_inicio_viagem TEXT,
        diferenca_chegada_motorista_fim_viagem BIGINT,
        data_fim_viagem TIMESTAMP,
        hora_fim_viagem TEXT,
        diferenca_inicio_viagem_fim_viagem BIGINT,
        quantidade_milhas_viagem DOUBLE PRECISION,
        quantidade_segundos_viagem DOUBLE PRECISION,
        valor_velocidade_media_viagem DOUBLE PRECISION,
        valor_tarifa_base_passageiro DOUBLE PRECISION,
        valor_pedagios DOUBLE PRECISION,
        valor_fundo_black_car DOUBLE PRECISION,
        valor_imposto_vendas DOUBLE PRECISION,
        valor_taxa_congestionamento DOUBLE PRECISION,
        valor_taxa_aeroporto DOUBLE PRECISION,
        valor_gorjeta DOUBLE PRECISION,
        valor_pagamento_motorista DOUBLE PRECISION,
        valor_receita_total_viagem DOUBLE PRECISION,
        indicador_corrida_compartilhada_solicitada TEXT,
        indicador_corrida_compartilhada_realizada TEXT,
        indicador_corrida_mta TEXT,
        indicador_veiculo_acessivel_solicitado TEXT,
        indicador_veiculo_acessivel_realizado TEXT,
        valor_taxa_congestionamento_cb DOUBLE PRECISION
    );
"""
)

#### 7.1. Realizar a inserção dos dados da fato viagem

In [15]:
con.execute(f"""
    INSERT INTO 
        pg.public.tb_spec_fat_viagem
    SELECT 
        identificador_empresa,
        identificador_base_despacho,
        identificador_base_origem,
        data_solicitacao,
        hora_solicitacao,
        data_chegada_motorista,
        hora_chegada_motorista,
        data_inicio_viagem,
        hora_inicio_viagem,
        diferenca_chegada_motorista_fim_viagem,
        data_fim_viagem,
        hora_fim_viagem,
        diferenca_inicio_viagem_fim_viagem,
        quantidade_milhas_viagem,
        quantidade_segundos_viagem,
        valor_velocidade_media_viagem,
        valor_tarifa_base_passageiro,
        valor_pedagios,
        valor_fundo_black_car,
        valor_imposto_vendas,
        valor_taxa_congestionamento,
        valor_taxa_aeroporto,
        valor_gorjeta,
        valor_pagamento_motorista,
        valor_receita_total_viagem,
        indicador_corrida_compartilhada_solicitada,
        indicador_corrida_compartilhada_realizada,
        indicador_corrida_mta,
        indicador_veiculo_acessivel_solicitado,
        indicador_veiculo_acessivel_realizado,
        valor_taxa_congestionamento_cb
    FROM 
        pg.public.tb_spec_viagem
"""
)

#### 8. Realizar as consultas no banco de dados para responder as perguntas de negócio

8.1. Qual a quantidade, receita e ticket médio (receita / quantidade) das viagens solicitadas por trimestre em 2025?

In [16]:
query_1 = con.execute(f"""
    SELECT 
        CASE 
            WHEN dim_data.anomes BETWEEN '2025-01' AND '2025-03' THEN '1° Trimestre / 2025'
            WHEN dim_data.anomes BETWEEN '2025-04' AND '2025-06' THEN '2° Trimestre / 2025'
            WHEN dim_data.anomes BETWEEN '2025-07' AND '2025-09' THEN '3° Trimestre / 2025'
            WHEN dim_data.anomes BETWEEN '2025-10' AND '2025-12' THEN '4° Trimestre / 2025'
        END AS trimestre,
        CAST(COUNT(*) / 1000000 AS DECIMAL(20, 2)) AS quantidade_milhao,
        CAST(SUM(valor_receita_total_viagem) / 1000000 AS DECIMAL(20,2)) AS receita_milhao,
        CAST(SUM(valor_receita_total_viagem) / COUNT(*) AS DECIMAL(20, 2)) AS ticket_medio
    FROM 
        pg.public.tb_spec_fat_viagem AS fat_viagem
    LEFT JOIN 
        pg.public.tb_spec_dim_data AS dim_data
        ON fat_viagem.data_solicitacao = dim_data.data
    GROUP BY 
        1
    ORDER BY 
        1;
"""
).df()

query_1

,trimestre,quantidade_milhao,receita_milhao,ticket_medio
0,1° Trimestre / 2025,3.02,97.25,32.25
1,2° Trimestre / 2025,3.05,105.35,34.56
2,3° Trimestre / 2025,2.93,100.29,34.20
3,4° Trimestre / 2025,3.21,110.35,34.42


8.2. Qual a distância, duração e velocidade média das viagens por empresa?

In [17]:
query_2 = con.execute(f"""
    SELECT 
        dim_empresa.nome_empresa AS empresa,
        CAST(AVG(fat_viagem.quantidade_milhas_viagem) AS DECIMAL(20, 2)) AS distancia_media_milha,
        CAST(AVG(fat_viagem.quantidade_segundos_viagem / 60) AS DECIMAL(20, 2)) AS duracao_media_minuto,
        CAST(AVG(fat_viagem.valor_velocidade_media_viagem) AS DECIMAL(20, 2)) AS velocidade_media_milha_minuto
    FROM 
        pg.public.tb_spec_fat_viagem AS fat_viagem
    LEFT JOIN 
        pg.public.tb_spec_dim_empresa AS dim_empresa
        ON fat_viagem.identificador_empresa = dim_empresa.identificador_empresa
    GROUP BY 
        1
    ORDER BY 
        1;
""").df()

query_2

,empresa,distancia_media_milha,duracao_media_minuto,velocidade_media_milha_minuto
0,Lyft,4.85,19.71,13.19
1,Uber,5.10,19.94,13.63


8.3. Como os indicadores de receita, lucro e lucratividade evoluem históricamente?

In [18]:
query_3 = con.execute(f"""
    SELECT
        dim_data.anomes AS periodo,
        CAST((SUM(fat_viagem.valor_receita_total_viagem - fat_viagem.valor_pagamento_motorista) / 1000000) AS DECIMAL(20, 2)) AS lucro_milhao,
        CAST((SUM(fat_viagem.valor_receita_total_viagem) / 1000000) AS DECIMAL(20, 2)) AS receita_milhao,
        CAST((SUM(fat_viagem.valor_receita_total_viagem - fat_viagem.valor_pagamento_motorista) / NULLIF(SUM(fat_viagem.valor_receita_total_viagem), 0)) * 100 AS DECIMAL(20, 2)) AS lucratividade
    FROM 
        pg.public.tb_spec_fat_viagem AS fat_viagem
    LEFT JOIN 
        pg.public.tb_spec_dim_data AS dim_data
        ON fat_viagem.data_solicitacao = dim_data.data
    GROUP BY 
        1
    ORDER BY 
        1;
""").df()

query_3

,periodo,lucro_milhao,receita_milhao,lucratividade
0,2025-01,12.36,31.46,39.29
1,2025-02,11.92,30.35,39.28
2,2025-03,14.52,35.44,40.96
3,2025-04,12.93,33.16,39.00
4,2025-05,14.34,37.35,38.40
5,2025-06,13.36,34.84,38.35
6,2025-07,12.46,32.88,37.90
7,2025-08,12.01,32.14,37.37
8,2025-09,13.68,35.26,38.78
9,2025-10,13.62,36.61,37.20


8.4. Das viagens acessíveis solicitadas, quantas foram de fato realizadas?

In [19]:
query_4 = con.execute(f"""
    SELECT 
        CAST(veiculo_acessivel_realizado AS BIGINT) AS realizado,
        CAST(veiculo_acessivel_solicitado AS BIGINT) AS solicitado,
        CAST((veiculo_acessivel_realizado / veiculo_acessivel_solicitado) * 100 AS DECIMAL(20, 2)) AS percentual
    FROM 
        (
            SELECT
                SUM(CASE WHEN indicador_veiculo_acessivel_solicitado = 'N' THEN 1 ELSE 0 END) AS veiculo_acessivel_solicitado,
                SUM(CASE WHEN indicador_veiculo_acessivel_realizado = 'N' THEN 1 ELSE 0 END) AS veiculo_acessivel_realizado
            FROM
                pg.public.tb_spec_fat_viagem
        )
""").df()

query_4

,realizado,solicitado,percentual
0,11130985,12166098,91.49


8.5. Como o custo das viagens varia de acordo com o dia da semana e período?

In [20]:
query_5 = con.execute(f"""
    SELECT 
        dim_data.anomes as periodo,
        dim_data.dia_semana,
        AVG(fat_viagem.valor_tarifa_base_passageiro) AS valor_medio
    FROM 
        pg.public.tb_spec_fat_viagem AS fat_viagem
    LEFT JOIN 
        pg.public.tb_spec_dim_data AS dim_data 
        ON fat_viagem.data_solicitacao = dim_data.data
    GROUP BY 
        1, 2
    ORDER BY 
        1 ASC, 2 ASC
""").df()

query_5

,periodo,dia_semana,valor_medio
0,2025-01,a. Domingo,23.195259
1,2025-01,b. Segunda-feira,24.380678
2,2025-01,c. Terça-feira,24.464011
3,2025-01,d. Quarta-feira,25.482074
4,2025-01,e. Quinta-feira,25.397779
...,...,...,...
79,2025-12,c. Terça-feira,27.823018
80,2025-12,d. Quarta-feira,28.658354
81,2025-12,e. Quinta-feira,30.988312
82,2025-12,f. Sexta-feira,27.396649


#### 9. Realizar a gravação dos arquivos parquet da camada SPEC

In [21]:
output_path = "../../data/2-spec/"

con.execute(f"""COPY (SELECT * FROM pg.public.tb_spec_dim_data) TO "{output_path}tb-spec-dim-data.parquet" (FORMAT "parquet", COMPRESSION "snappy");""")

con.execute(f"""COPY (SELECT * FROM pg.public.tb_spec_dim_empresa) TO "{output_path}tb-spec-dim-empresa.parquet" (FORMAT "parquet", COMPRESSION "snappy");""")

con.execute(f"""COPY (SELECT * FROM pg.public.tb_spec_dim_flag) TO "{output_path}tb-spec-dim-flag.parquet" (FORMAT "parquet", COMPRESSION "snappy");""")

con.execute(f"""COPY (SELECT * FROM pg.public.tb_spec_dim_hora) TO "{output_path}tb-spec-dim-hora.parquet" (FORMAT "parquet", COMPRESSION "snappy");""")

con.execute(f"""COPY (SELECT * FROM pg.public.tb_spec_fat_viagem) TO "{output_path}tb-spec-fat-viagem.parquet" (FORMAT "parquet", COMPRESSION "snappy");""")

#### 10. Realiza o encerramento da sessão com o banco de dados

In [22]:
con.close()